# Morgan(1983) 거리감쇄 기반 상호작용 지수(PC*) — Python 구현

**논문:** Morgan, B.S. (1983). "A distance-decay based interaction index to measure residential segregation." *Area*, 15(3), 211–217.

---

## 이 노트북의 구성

| 섹션 | 내용 |
|------|------|
| 1 | 라이브러리 설치 및 불러오기 |
| 2 | 기본 P* 지수 구현 |
| 3 | 거리감쇄 함수 및 접촉률 행렬 Cij |
| 4 | 상호작용 확률 행렬 Pij |
| 5 | PC* 지수 (논문 핵심 수식) |
| 6 | 표준화 지수 I1, I2, IC1, IC2 |
| 7 | 가상 도시 데이터 데모 |
| 8 | 결과 시각화 |

---

## 핵심 수식 요약

**기본 P\* (Lieberson 1981):**
$$_aP^*_b = \sum_i \frac{a_i}{\sum_i a_i} \cdot \frac{b_i}{t_i}$$

**PC\* (Morgan 1983):**
$$_aPC^*_b = \sum_i \frac{a_i}{\sum_i a_i} \left[ \sum_j P_{ij} \cdot \frac{b_j}{t_j} \right]$$

**거리감쇄 함수:**
$$\log C_{ij} = a - b \cdot d_{ij}^m, \quad m \leq 0.5$$

**표준화 지수:**
$$I_2 = \frac{_aP^*_b}{B}, \qquad IC_2 = \frac{_aPC^*_b}{B}$$

## 섹션 1: 라이브러리 설치 및 불러오기

### 필요한 라이브러리 안내

- **numpy:** 행렬 연산(행렬 곱셈, 원소별 연산 등)을 위해 사용
- **pandas:** 표 형태의 데이터를 보기 좋게 출력하기 위해 사용
- **matplotlib:** 그래프 시각화를 위해 사용

### 설치 방법 (아직 없다면 터미널에서 실행)

```
pip install numpy pandas matplotlib
```

In [ ]:
# [목적]
#   - 이 셀을 가장 먼저 실행해야 함
#   - 이후 모든 계산에서 사용할 라이브러리를 불러옴
#
# [라이브러리 설명]
#   - import numpy as np   : numpy를 'np'라는 별칭으로 불러옴
#                            이후 np.array(), np.sum() 등으로 사용
#   - import pandas as pd  : pandas를 'pd'라는 별칭으로 불러옴
#                            이후 pd.DataFrame() 등으로 사용
#   - import matplotlib    : 그래프 그리기용
#   - %matplotlib inline   : Jupyter 노트북 내에서 그래프를 바로 표시

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'  # macOS 한글 폰트
matplotlib.rcParams['axes.unicode_minus'] = False    # 마이너스 기호 깨짐 방지
%matplotlib inline

print("라이브러리 로드 완료")
print(f"  numpy 버전:      {np.__version__}")
print(f"  pandas 버전:     {pd.__version__}")
print(f"  matplotlib 버전: {matplotlib.__version__}")

---

## 섹션 2: 기본 P\* 지수

### 수식

$$_aP^*_b = \sum_i \frac{a_i}{\sum_i a_i} \cdot \frac{b_i}{t_i}$$

### 변수 설명

| 기호 | 의미 |
|------|------|
| $a_i$ | i번째 소구역(tract)의 집단 A 인구 수 |
| $b_i$ | i번째 소구역의 집단 B 인구 수 |
| $t_i$ | i번째 소구역의 총 인구 수 |

### 해석

- A 집단이 자신의 소구역 내에서 B 집단을 만날 확률
- 0에 가까울수록 분리, 1에 가까울수록 혼합

In [ ]:
def compute_P_star(a, b, t):
    """
    기본 P* 지수 계산 함수.

    [수식]
        aPb* = sum_i [ (a_i / sum(a)) * (b_i / t_i) ]

    [입력 인수]
        - a : 각 소구역의 집단 A 인구 수 (1차원 배열 또는 리스트)
        - b : 각 소구역의 집단 B 인구 수 (1차원 배열 또는 리스트)
        - t : 각 소구역의 총 인구 수     (1차원 배열 또는 리스트)

    [출력]
        - 실수(float) 1개: 0과 1 사이의 상호작용 확률
    """
    # [단계 1] numpy 배열로 변환
    #   - np.array() : 리스트나 다른 형태를 numpy 배열로 변환
    #   - dtype=float : 정수 나눗셈에서 소수점 손실을 막기 위해 실수형 지정
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    t = np.array(t, dtype=float)

    # [단계 2] 입력값 검증
    #   - a.sum() : a 배열의 모든 원소를 합산
    total_a = a.sum()
    if total_a == 0:
        raise ValueError("집단 A의 전체 인구가 0입니다.")

    # [단계 3] 핵심 계산
    #   - a / total_a : 각 구역 A 인구의 전체 대비 비중 (가중치)
    #   - b / t       : 각 구역의 B 집단 비율
    #   - * 연산자    : numpy 배열끼리는 원소별(element-wise) 곱셈
    #   - np.sum()    : 배열의 모든 원소를 합산
    return np.sum((a / total_a) * (b / t))


# --- 간단한 테스트 ---
# [테스트 데이터]
#   - 2개 구역, 완전 분리 상황
#   - 구역1: A=100, B=0, 총=100
#   - 구역2: A=0,   B=100, 총=100
#   - 기댓값: 0.0 (A와 B가 완전히 분리)
a_test = [100, 0]
b_test = [0, 100]
t_test = [100, 100]
result = compute_P_star(a_test, b_test, t_test)
print(f"[완전 분리] P* = {result:.4f}  (기댓값: 0.0)")

# [테스트 데이터 2]
#   - 2개 구역, 완전 혼합 상황
#   - 구역1: A=50, B=50, 총=100
#   - 구역2: A=50, B=50, 총=100
#   - 기댓값: 0.5 (B 집단이 전체 인구의 50%)
a_test2 = [50, 50]
b_test2 = [50, 50]
t_test2 = [100, 100]
result2 = compute_P_star(a_test2, b_test2, t_test2)
print(f"[완전 혼합] P* = {result2:.4f}  (기댓값: 0.5)")

---

## 섹션 3: 거리감쇄 함수 및 접촉률 행렬 Cij

### 수식 (Taylor 1971 single-log 모형)

$$\log_{10} C_{ij} = a - b \cdot d_{ij}^m, \quad m \leq 0.5$$

$$\therefore \quad C_{ij} = 10^{\, a - b \cdot d_{ij}^m}$$

### 변수 설명

| 기호 | 의미 |
|------|------|
| $C_{ij}$ | tract i와 zone j 사이의 접촉률 (zone j 인구 1,000명당 접촉 건수) |
| $d_{ij}$ | tract i 중심에서 zone j 중심까지의 거리 (km) |
| $a$ | 절편 상수 (거리 0에서의 기준 접촉률) |
| $b$ | 기울기 상수 (거리 증가에 따른 감소 속도) |
| $m$ | 거리 변환 지수 (0.5 이하 권고) |

In [ ]:
def compute_C_matrix(dist_matrix, a_param, b_param, m=0.5):
    """
    거리감쇄 함수로 접촉률 행렬 Cij를 계산.

    [수식]
        log10(C_ij) = a_param - b_param * d_ij^m
        C_ij = 10^(a_param - b_param * d_ij^m)

    [입력 인수]
        - dist_matrix : n_tracts x n_zones 거리 행렬 (2차원 배열)
        - a_param     : 절편 상수
        - b_param     : 기울기 상수
        - m           : 거리 변환 지수 (기본값 0.5)

    [출력]
        - n_tracts x n_zones 크기의 접촉률 행렬
    """
    # [단계 1] numpy 배열로 변환
    dist_matrix = np.array(dist_matrix, dtype=float)

    # [단계 2] 입력값 검증
    if m > 0.5:
        print(f"경고: m={m} > 0.5. 논문 권고(m<=0.5)를 초과합니다.")
    if np.any(dist_matrix < 0):
        raise ValueError("거리 행렬에 음수값이 있습니다.")

    # [단계 3] 핵심 계산
    #   - dist_matrix ** m : 각 거리 원소에 m 제곱을 적용
    #                        m=0.5이면 제곱근(sqrt)과 동일
    #   - 10 ** (...) : 10의 거듭제곱 = 상용로그의 역함수
    #                   즉, log10(C) = x 이면 C = 10^x
    log_C = a_param - b_param * (dist_matrix ** m)
    return 10 ** log_C


# --- 거리감쇄 곡선 시각화 ---
# [목적]
#   - 거리에 따라 접촉률이 어떻게 변하는지 그래프로 확인
#   - 논문 Table 1의 패턴(근거리 급격 감소)과 일치하는지 검증
distances = np.linspace(0.1, 10, 200)  # 0.1km부터 10km까지 200개 구간

# 파라미터 조합 3가지 비교
fig, ax = plt.subplots(figsize=(9, 5))

for m_val, color, label in [(0.5, 'steelblue', 'm=0.5 (권고값)'),
                             (0.3, 'darkorange', 'm=0.3'),
                             (0.7, 'firebrick', 'm=0.7 (초과, 참고용)')]:
    C_vals = 10 ** (2.0 - 0.5 * distances ** m_val)
    ax.plot(distances, C_vals, color=color, lw=2, label=label)

ax.set_xlabel('거리 (km)', fontsize=12)
ax.set_ylabel('접촉률 Cij (1,000명당)', fontsize=12)
ax.set_title('거리감쇄 함수: log(Cij) = 2.0 - 0.5 * d^m', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print("\n거리별 접촉률 예시 (a=2.0, b=0.5, m=0.5):")
ex_dists = [0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0]
for d in ex_dists:
    c = 10 ** (2.0 - 0.5 * d ** 0.5)
    print(f"  거리 {d:5.1f}km -> Cij = {c:.4f}")

---

## 섹션 4: 상호작용 확률 행렬 Pij

### 수식

$$P_{ij} = \frac{C_{ij} \cdot t_j}{\sum_j C_{ij} \cdot t_j}, \qquad \text{단, } \sum_j P_{ij} = 1.0$$

### 해석

- $P_{ij}$: tract i 거주자의 다음 접촉 상대가 zone j에 있을 확률
- 두 가지 요소를 동시에 반영
  - 거리 요소: zone j가 tract i와 가까울수록($C_{ij}$ 큼) 확률 증가
  - 규모 요소: zone j의 인구($t_j$)가 많을수록 확률 증가
- 분모로 나누어 정규화 → 모든 zone의 확률 합 = 1.0

In [ ]:
def compute_P_ij(C_matrix, pop_zones):
    """
    상호작용 확률 행렬 Pij를 계산.

    [수식]
        P_ij = (C_ij * t_j) / sum_j(C_ij * t_j)
        조건: sum_j P_ij = 1.0 for all i

    [입력 인수]
        - C_matrix  : compute_C_matrix()의 출력 (n_tracts x n_zones 배열)
        - pop_zones : 각 zone j의 총 인구 수 (1차원 배열, 길이 n_zones)

    [출력]
        - n_tracts x n_zones 크기의 확률 행렬
        - 각 행(tract)의 합 = 1.0
    """
    # [단계 1] numpy 배열로 변환
    C_matrix = np.array(C_matrix, dtype=float)
    pop_zones = np.array(pop_zones, dtype=float)

    # [단계 2] 입력 크기 검증
    #   - C_matrix.shape[1] : 행렬의 열 수 (n_zones)
    if C_matrix.shape[1] != len(pop_zones):
        raise ValueError("C_matrix의 열 수와 pop_zones의 길이가 다릅니다.")

    # [단계 3] C_ij * t_j 계산
    #   - C_matrix * pop_zones : numpy 브로드캐스팅
    #     C_matrix(n_tracts x n_zones) * pop_zones(n_zones,)
    #     => 각 행(tract i)의 모든 열(zone j)에 pop_zones[j]를 곱함
    #     결과: n_tracts x n_zones 행렬
    weighted = C_matrix * pop_zones

    # [단계 4] 행별 합산: sum_j(C_ij * t_j)
    #   - weighted.sum(axis=1) : 각 행의 합 계산
    #                            axis=0은 열 방향, axis=1은 행 방향
    #   - keepdims=True : 결과를 (n_tracts, 1) 형태로 유지 (나눗셈 브로드캐스팅용)
    row_sums = weighted.sum(axis=1, keepdims=True)  # (n_tracts, 1)

    # [단계 5] 정규화: P_ij = (C_ij * t_j) / sum_j(C_ij * t_j)
    #   - weighted / row_sums : (n_tracts x n_zones) / (n_tracts x 1)
    #                           브로드캐스팅으로 각 행을 자신의 합으로 나눔
    return weighted / row_sums


# --- 간단한 검증 ---
# [테스트]
#   - 2개 tract, 3개 zone
#   - 모든 행의 합이 1.0인지 확인
C_test = np.array([[10.0, 5.0, 2.0],
                   [ 2.0, 5.0, 10.0]])
pop_test = np.array([100.0, 200.0, 100.0])
P_test = compute_P_ij(C_test, pop_test)

print("[Pij 검증]")
print(f"  P_ij 행렬:\n{np.round(P_test, 4)}")
print(f"  각 행의 합: {P_test.sum(axis=1).round(10)}  (모두 1.0이어야 함)")
print()
print("  해석:")
print("  - Tract 1(왼쪽 근거리)의 P_ij: zone 1 방문 확률이 가장 높음")
print("  - Tract 2(오른쪽 근거리)의 P_ij: zone 3 방문 확률이 가장 높음")

---

## 섹션 5: PC\* 지수 (논문 핵심 수식)

### 수식

$$_aPC^*_b = \sum_i \frac{a_i}{\sum_i a_i} \left[ \sum_j P_{ij} \cdot \frac{b_j}{t_j} \right]$$

### P\*와의 차이

| 구분 | P\* | PC\* |
|------|-----|------|
| 접촉 범위 | 동일 소구역(tract) 내 | 도시 전역(city-wide) |
| 거리 반영 | 없음 | 거리감쇄 함수로 반영 |
| 계산 기준 | $b_i/t_i$ (소구역 B 비율) | $\sum_j P_{ij}(b_j/t_j)$ (도시 전역 가중 B 비율) |

In [ ]:
def compute_PC_star(a_tracts, b_zones, t_zones, P_ij):
    """
    PC* 지수 계산 함수 (Morgan 1983 핵심 수식).

    [수식]
        aPCb* = sum_i [(a_i / sum(a)) * sum_j(P_ij * b_j/t_j)]

    [입력 인수]
        - a_tracts : 각 tract i의 집단 A 인구 수 (1차원 배열, 길이 n_tracts)
        - b_zones  : 각 zone j의 집단 B 인구 수  (1차원 배열, 길이 n_zones)
        - t_zones  : 각 zone j의 총 인구 수      (1차원 배열, 길이 n_zones)
        - P_ij     : compute_P_ij()의 출력 (n_tracts x n_zones 배열)

    [출력]
        - 실수(float) 1개: 0과 1 사이의 도시 전역 상호작용 확률
    """
    # [단계 1] numpy 배열 변환
    a_tracts = np.array(a_tracts, dtype=float)
    b_zones  = np.array(b_zones,  dtype=float)
    t_zones  = np.array(t_zones,  dtype=float)
    P_ij     = np.array(P_ij,     dtype=float)

    # [단계 2] 입력값 검증
    total_a = a_tracts.sum()
    if total_a == 0:
        raise ValueError("집단 A의 전체 인구가 0입니다.")

    # [단계 3] b_j / t_j : 각 zone의 B 집단 비율
    b_ratio = b_zones / t_zones  # 1차원 배열 (n_zones,)

    # [단계 4] sum_j P_ij * (b_j/t_j) : 각 tract별 도시 전역 B 접촉 확률
    #   - P_ij @ b_ratio : 행렬-벡터 곱셈 (@ 연산자)
    #                      P_ij(n_tracts x n_zones) @ b_ratio(n_zones,)
    #                      결과: (n_tracts,) 벡터
    #   - inner_sum[i] = tract i에서 도시 전역 B를 만날 확률
    inner_sum = P_ij @ b_ratio  # (n_tracts,)

    # [단계 5] 최종 가중 합산
    #   - a_tracts / total_a : 각 tract의 A 집단 비중 (가중치)
    #   - np.dot() : 벡터 내적 = Σ_i (a_i/Σa_i) * inner_sum[i]
    return np.dot(a_tracts / total_a, inner_sum)

---

## 섹션 6: 표준화 지수 I1, I2, IC1, IC2

### 수식 (Bell 1954 방식)

$$I_1 = \frac{_aP^*_a - A}{1 - A} = 1.0 - I_2$$

$$I_2 = \frac{_aP^*_b}{B} = 1.0 - I_1$$

$$IC_2 = \frac{_aPC^*_b}{B} = 1.0 - IC_1$$

### 해석

| 지수 | 값 범위 | 해석 |
|------|---------|------|
| $I_1$ | 0 ~ 1 | 0=완전혼합, 1=완전고립 |
| $I_2$ | 0 ~ ? | 1.0=무작위 혼합, <1=분리, >1=과혼합 |
| $IC_2$ | 0 ~ ? | I₂와 동일 해석 (도시 전역 기준) |

In [ ]:
def compute_I1(P_aa, A):
    """
    고립 지수(Isolation Index) I1 계산.

    [수식]  I1 = (aPa* - A) / (1 - A)

    [입력 인수]
        - P_aa : compute_P_star(a, a, t)의 결과
                 A 집단이 같은 A 집단을 만날 확률
        - A    : 전체 인구 중 집단 A의 비율 (= sum(a) / sum(t))

    [출력]
        - 실수: 0과 1 사이의 고립 지수
    """
    # [분모 검증]
    #   - 1 - A == 0 이면 전체 인구가 A 집단뿐 → 계산 불가
    if A >= 1.0:
        raise ValueError("A 비율이 1.0 이상입니다.")
    return (P_aa - A) / (1 - A)


def compute_I2(P_ab, B):
    """
    집단 분리 비율(Group Segregation Ratio) I2 계산.

    [수식]  I2 = aPb* / B

    [입력 인수]
        - P_ab : compute_P_star(a, b, t)의 결과
                 A 집단이 B 집단을 만날 확률
        - B    : 전체 인구 중 집단 B의 비율 (= sum(b) / sum(t))

    [출력]
        - 실수: 집단 분리 비율 (1.0=무작위 혼합)
    """
    if B == 0:
        raise ValueError("B 비율이 0입니다.")
    return P_ab / B


def compute_IC2(PC_ab, B):
    """
    도시 전역 표준화 지수 IC2 계산.

    [수식]  IC2 = aPCb* / B

    [입력 인수]
        - PC_ab : compute_PC_star()의 결과 (도시 전역 상호작용 확률)
        - B     : 전체 인구 중 집단 B의 비율

    [출력]
        - 실수: 도시 전역 집단 분리 비율
    """
    if B == 0:
        raise ValueError("B 비율이 0입니다.")
    return PC_ab / B


def compute_IC1(IC2):
    """
    IC1 = 1.0 - IC2

    [수식]  IC1 = 1.0 - IC2

    [입력 인수]
        - IC2 : compute_IC2()의 결과

    [출력]
        - 실수
    """
    return 1.0 - IC2


print("표준화 지수 함수 정의 완료: compute_I1, compute_I2, compute_IC2, compute_IC1")

---

## 섹션 7: 가상 도시 데이터 데모 — 전체 계산 실행

### 가상 도시 설정

- 5개 tract(소구역)으로 구성된 가상 도시
- 집단 A = 백인(White), 집단 B = 흑인(Black)
- 북서쪽에 A 집단 집중, 남동쪽에 B 집단 집중 (분리 상황 가정)
- tract = zone (동일 공간 단위 사용)
- 거리감쇄 파라미터: a=2.0, b=0.5, m=0.5

In [ ]:
# ============================================================
# 데이터 설정
# ============================================================

# [인구 데이터]
#   - tract_names : 각 소구역의 이름 (리스트)
#   - a_pop       : 각 소구역의 집단 A (백인) 인구 수
#   - b_pop       : 각 소구역의 집단 B (흑인) 인구 수
#   - t_pop       : 각 소구역의 총 인구 수 (A + B + 기타)
#
# Tract 1~2: 북서쪽, A 집단 우세
# Tract 3  : 도심, 혼합 지역
# Tract 4~5: 남동쪽, B 집단 우세

tract_names = ['Tract1(NW)', 'Tract2(N)', 'Tract3(C)', 'Tract4(S)', 'Tract5(SE)']
a_pop = np.array([180, 150,  50,  30,  20], dtype=float)
b_pop = np.array([ 10,  20,  80, 140, 160], dtype=float)
t_pop = np.array([200, 200, 200, 200, 200], dtype=float)

# [인구 데이터 확인]
pop_df = pd.DataFrame({
    '구역':    tract_names,
    '집단A':   a_pop.astype(int),
    '집단B':   b_pop.astype(int),
    '총인구':  t_pop.astype(int),
    'A비율':   (a_pop / t_pop).round(3),
    'B비율':   (b_pop / t_pop).round(3)
})
print("[인구 데이터]")
print(pop_df.to_string(index=False))
print()

# [거리 행렬]
#   - 5x5 행렬: tract i에서 zone j(=tract j)까지의 거리(km)
#   - 대각선: 0.5km (같은 tract 내부 거리, 0 대신 작은 양수 사용)
#   - 논문 Table 1처럼 근거리 접촉이 가장 강한 패턴 반영
dist_matrix = np.array([
    [0.5, 1.2, 2.5, 4.0, 5.5],  # Tract1(NW) 기준 거리
    [1.2, 0.5, 1.5, 3.0, 4.5],  # Tract2(N)  기준 거리
    [2.5, 1.5, 0.5, 1.5, 2.5],  # Tract3(C)  기준 거리 (도심, 균등)
    [4.0, 3.0, 1.5, 0.5, 1.2],  # Tract4(S)  기준 거리
    [5.5, 4.5, 2.5, 1.2, 0.5],  # Tract5(SE) 기준 거리
], dtype=float)

dist_df = pd.DataFrame(dist_matrix, index=tract_names, columns=tract_names)
print("[거리 행렬 (km)]")
print(dist_df.round(2).to_string())
print()

# [거리감쇄 파라미터]
a_param = 2.0   # 절편 상수
b_param = 0.5   # 기울기 상수
m       = 0.5   # 거리 변환 지수 (권고값 <= 0.5)

print(f"[거리감쇄 파라미터] a={a_param}, b={b_param}, m={m}")

In [ ]:
# ============================================================
# 계산 1: 기본 P* 지수
# ============================================================
print("=" * 55)
print("[계산 1] 기본 P* 지수")
print("=" * 55)

# [A→B 상호작용]
#   - A 집단이 같은 소구역 안에서 B 집단을 만날 확률
P_star_ab = compute_P_star(a_pop, b_pop, t_pop)

# [B→A 상호작용]
#   - B 집단이 같은 소구역 안에서 A 집단을 만날 확률
P_star_ba = compute_P_star(b_pop, a_pop, t_pop)

# [자기 집단 내 상호작용]
#   - compute_P_star(a, a, t) : b 자리에 a를 넣으면 자기 집단 내 확률
P_star_aa = compute_P_star(a_pop, a_pop, t_pop)
P_star_bb = compute_P_star(b_pop, b_pop, t_pop)

print(f"  aPb*  (A가 소구역 내 B를 만날 확률) = {P_star_ab:.4f}")
print(f"  bPa*  (B가 소구역 내 A를 만날 확률) = {P_star_ba:.4f}")
print(f"  aPa*  (A가 소구역 내 A를 만날 확률) = {P_star_aa:.4f}")
print(f"  bPb*  (B가 소구역 내 B를 만날 확률) = {P_star_bb:.4f}")
print()

# ============================================================
# 계산 2: 접촉률 행렬 Cij + 상호작용 확률 행렬 Pij
# ============================================================
print("=" * 55)
print("[계산 2] 접촉률/확률 행렬 계산")
print("=" * 55)

C_matrix = compute_C_matrix(dist_matrix, a_param, b_param, m)
print("  Cij 행렬 (1,000명당 접촉 건수):")
C_df = pd.DataFrame(C_matrix.round(3), index=tract_names, columns=tract_names)
print(C_df.to_string())
print()

P_ij = compute_P_ij(C_matrix, t_pop)
print("  Pij 행렬 (각 zone 방문 확률):")
P_df = pd.DataFrame(P_ij.round(4), index=tract_names, columns=tract_names)
print(P_df.to_string())
print(f"\n  행별 합 = {P_ij.sum(axis=1).round(10)}  (모두 1.0이어야 함)")
print()

# ============================================================
# 계산 3: PC* 지수
# ============================================================
print("=" * 55)
print("[계산 3] PC* 지수 (도시 전역 상호작용)")
print("=" * 55)

PC_star_ab = compute_PC_star(a_pop, b_pop, t_pop, P_ij)
PC_star_ba = compute_PC_star(b_pop, a_pop, t_pop, P_ij)
PC_star_aa = compute_PC_star(a_pop, a_pop, t_pop, P_ij)
PC_star_bb = compute_PC_star(b_pop, b_pop, t_pop, P_ij)

print(f"  aPCb* (A가 도시 전역에서 B를 만날 확률) = {PC_star_ab:.4f}")
print(f"  bPCa* (B가 도시 전역에서 A를 만날 확률) = {PC_star_ba:.4f}")
print(f"  aPCa* (A가 도시 전역에서 A를 만날 확률) = {PC_star_aa:.4f}")
print(f"  bPCb* (B가 도시 전역에서 B를 만날 확률) = {PC_star_bb:.4f}")
print()

# ============================================================
# 계산 4: 표준화 지수
# ============================================================
print("=" * 55)
print("[계산 4] 표준화 지수")
print("=" * 55)

A_ratio = a_pop.sum() / t_pop.sum()  # 전체 인구 중 A 집단 비율
B_ratio = b_pop.sum() / t_pop.sum()  # 전체 인구 중 B 집단 비율

I1  = compute_I1(P_star_aa, A_ratio)
I2  = compute_I2(P_star_ab, B_ratio)
IC2 = compute_IC2(PC_star_ab, B_ratio)
IC1 = compute_IC1(IC2)

print(f"  A 집단 전체 비율 (A) = {A_ratio:.4f}")
print(f"  B 집단 전체 비율 (B) = {B_ratio:.4f}")
print()
print(f"  I1  (고립 지수,           P* 기반)  = {I1:.4f}")
print(f"  I2  (집단 분리 비율,      P* 기반)  = {I2:.4f}")
print(f"  IC2 (도시전역 분리 비율, PC* 기반)  = {IC2:.4f}")
print(f"  IC1 = 1 - IC2                      = {IC1:.4f}")
print(f"\n  검증: I1 + I2 = {I1+I2:.6f}  (1.0이어야 함)")

---

## 섹션 8: 결과 시각화

In [ ]:
# ============================================================
# 그림 1: 소구역별 인구 구성 막대그래프
# ============================================================
# [목적]
#   - 5개 tract의 A/B 집단 분포를 시각적으로 확인
#   - 북서쪽 A 집중, 남동쪽 B 집중 패턴 확인

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- 왼쪽: 인구 구성 막대그래프 ---
x = np.arange(len(tract_names))  # x축 위치 (0,1,2,3,4)
width = 0.35                       # 막대 너비

# [막대 그리기]
#   - axes[0].bar() : 막대그래프 그리기
#   - x - width/2   : A 집단 막대를 왼쪽으로 이동
#   - x + width/2   : B 집단 막대를 오른쪽으로 이동
axes[0].bar(x - width/2, a_pop, width, label='집단 A (백인)', color='steelblue', alpha=0.8)
axes[0].bar(x + width/2, b_pop, width, label='집단 B (흑인)', color='firebrick', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(tract_names, rotation=15, fontsize=9)
axes[0].set_ylabel('인구 수', fontsize=11)
axes[0].set_title('소구역별 집단 A/B 인구 분포', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.4)

# --- 오른쪽: 지수 비교 막대그래프 ---
index_names = ['P* (aPb*)', 'PC* (aPCb*)', 'I2', 'IC2']
index_values = [P_star_ab, PC_star_ab, I2, IC2]
colors = ['steelblue', 'darkorange', 'teal', 'purple']

bars = axes[1].bar(index_names, index_values, color=colors, alpha=0.8, width=0.5)
axes[1].axhline(y=1.0, color='gray', linestyle='--', linewidth=1.2, label='I2=1.0 (무작위 혼합 기준)')
axes[1].set_ylabel('지수 값', fontsize=11)
axes[1].set_title('Morgan(1983) 지수 비교', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.4)

# [값 표시]
#   - bar.get_height() : 각 막대의 높이(값)를 가져옴
for bar, val in zip(bars, index_values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

# ============================================================
# 그림 2: Pij 히트맵 - tract에서 zone 방문 확률
# ============================================================
fig2, ax2 = plt.subplots(figsize=(7, 5))

# [imshow] : 2차원 배열을 색깔 이미지로 표시
#   - cmap='YlOrRd' : 노란색(낮은값) → 빨간색(높은값) 색상 팔레트
im = ax2.imshow(P_ij, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax2, label='방문 확률 Pij')
ax2.set_xticks(range(5))
ax2.set_yticks(range(5))
ax2.set_xticklabels(tract_names, rotation=15, fontsize=9)
ax2.set_yticklabels(tract_names, fontsize=9)
ax2.set_xlabel('Zone j (방문지)', fontsize=11)
ax2.set_ylabel('Tract i (출발지)', fontsize=11)
ax2.set_title('상호작용 확률 행렬 Pij (거리감쇄 반영)', fontsize=12)

# [셀 내부에 수치 표시]
for i in range(5):
    for j in range(5):
        ax2.text(j, i, f'{P_ij[i,j]:.3f}',
                 ha='center', va='center', fontsize=9,
                 color='black' if P_ij[i,j] < 0.3 else 'white')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 최종 결과 요약 및 해석
# ============================================================
print("=" * 60)
print("  Morgan(1983) PC* 지수 — 최종 결과 요약")
print("=" * 60)

summary = pd.DataFrame({
    '지수':  ['P*',        'PC*',       'I2',             'IC2'],
    '값':    [round(P_star_ab, 4),
              round(PC_star_ab, 4),
              round(I2, 4),
              round(IC2, 4)],
    '기준':  ['0=분리, 1=혼합',
              '0=분리, 1=혼합',
              '1.0=무작위, <1=분리',
              '1.0=무작위, <1=분리'],
    '접촉범위': ['소구역 내',
                '도시 전역',
                '소구역 내(표준화)',
                '도시 전역(표준화)']
})

print(summary.to_string(index=False))
print()
print("[해석]")
print(f"  - P*  = {P_star_ab:.4f}: 소구역 수준에서 A가 B를 만날 확률")
print(f"  - PC* = {PC_star_ab:.4f}: 도시 전역에서 A가 B를 만날 확률")
delta = PC_star_ab - P_star_ab
if delta > 0:
    print(f"  - PC* > P* (차이: +{delta:.4f}): 도시 전역으로 확장하면 접촉 기회 증가")
else:
    print(f"  - PC* < P* (차이: {delta:.4f}): 도시 전역으로 확장해도 분리 유지")
print()
print(f"  - I2  = {I2:.4f} < 1.0: 소구역 수준 분리 존재")
print(f"  - IC2 = {IC2:.4f} < 1.0: 도시 전역 수준 분리도 존재")